# Trinity Hippocampal Learning Probe (THLP) - Benchmark Task

**DeepMind AGI Hackathon 2026**

This task evaluates hippocampal learning through error-driven belief updating.

**Key concepts** 
1. Task: A Python function defining the problem (hippocampal learning probe)
2. Run: The execution of a task
3. Benchmark: A collection of tasks for learning evaluation

In [ ]:
# Install kaggle-benchmarks (required for Benchmark Tasks)
!pip install -q kaggle-benchmarks

In [ ]:
# We import the library as 'kbench' for brevity
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass

print("Ready to benchmark!")

## Part 1: Creating Your First Task

In [ ]:
@kbench.task(name="thlp_single_item")
def thlp_single_item(llm, question: str, answer: str) -> dict:
    # 1. Prompt the LLM
    response = llm.prompt(question)
    print(f"Model Answer: {response}")

    # 2. Grade the response (simple string check)
    is_correct = answer.lower() in response.lower()

    # 3. Assert based on the boolean calculation
    kbench.assertions.assert_true(
        is_correct,
        expectation=f"The model's answer should contain '{answer}'."
    )

    # 4. Set a return value (optional, but useful for batch evaluation)
    return {
        "is_correct": is_correct,
        "model_response": response
    }

# Run the task immediately to test it
# thlp_single_item.run(
#     llm=kbench.llm,
#     question="A causes B. B occurs. Did A cause B?",
#     answer="Not necessarily",
# )

## Part 2: Scaling Up (Batch Evaluation)

In [ ]:
# 1. Create a small dataset with THLP learning probes
df = pd.DataFrame([
    {"question": "A causes B. B occurs. Did A cause B?", "answer": "not necessarily"},
    {"question": "If all bloops are bleeps and this is a bloop, is it a bleep?", "answer": "yes"},
    {"question": "What would happen if gravity suddenly stopped working for 5 seconds?", "answer": "float"},
    {"question": "A bird's wing is to flying as a fish's fin is to what?", "answer": "swimming"},
    {"question": "After failing to solve a puzzle twice, what strategy might help?", "answer": "different approach"},
    {"question": "If it rains, the ground gets wet. The ground is wet. Did it rain?", "answer": "not necessarily"},
    {"question": "What's the opposite of 'always'?", "answer": "never"},
    {"question": "If you flip a fair coin 10 times and get heads each time, what's the probability of heads on flip 11?", "answer": "0.5"}
])

print(f"Loaded {len(df)} THLP probes")

In [ ]:
# 2. Define a scoring task (returns an accuracy score)
@kbench.task(name="thlp_batch_accuracy")
def score_thlp_accuracy(llm, df) -> float:
    # Enable caching to speed up development and avoid re-running identical queries
    with kbench.client.enable_cache():
        # Execute the 'thlp_single_item' task for every row in our dataframe
        runs = thlp_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],  # Ensure the evaluation runs until all rows in the dataframe are processed
            max_attempts=1, # Limit retries to 1 to fail fast during testing
            llm=[llm], # Pass the specific LLM we want to evaluate
            evaluation_data=df,
            n_jobs=3, # Run 3 examples in parallel to significantly speed up the benchmark
        )

    # Convert the raw run objects into a pandas DataFrame for easy analysis
    eval_df = runs.as_dataframe()

    # Calculate the average success rate by taking the mean of the 'is_correct' column
    accuracy = float(eval_df.result.str.get("is_correct").mean())
    # Return the final calculated accuracy
    return accuracy

In [ ]:
# Run evaluation manually (uncomment to test)
# _ = score_thlp_accuracy.run(kbench.llm, df)

Congratulations! You've now run your first task over a dataset.

## Part 3: Choose the Task for your Task Detail page

Kaggle Benchmarks requires you to specify one primary task to populate your Task Detail page, which is created when you hit "Save Task" on the top right hand corner of this notebook.

Run the cell below to lock in `thlp_batch_accuracy` (instead of `thlp_single_item`) as your submitted task. You can change this later by pointing %choose to a different task function.

In [ ]:
%choose thlp_batch_accuracy

## (Optional) Part 4: Advanced Features
Now that you have the basics, here are powerful features to create more types of tasks.
- A. Complex Inputs (Vision, Multi-turn)
- B. Advanced Logic (Agents/Tools, Multi-Model Comparison)
- C. Deep Evaluation (Return Types, LLM-as-a-Judge)